# range, enumerate and zip

**Topic:** `2.5` &nbsp;|&nbsp; **Module 2: Control Flow and Loops**

*Three built-ins that remove the bookkeeping loops force on you, and the one silent failure each of them can still produce.*

> *Course:* Python Master Course &nbsp;|&nbsp; **Project:** ROBO-X (M2)

### Navigate

* [Lesson](lesson.ipynb) &nbsp;|&nbsp; [Exercises](exercises.ipynb) &nbsp;|&nbsp; [Solutions](solution.ipynb) &nbsp;|&nbsp; [Research](research.ipynb) &nbsp;|&nbsp; [Quiz](quiz.ipynb) &nbsp;|&nbsp; [Mini-project](mini_project.ipynb) &nbsp;|&nbsp; [Robotics challenge](robotics_challenge.ipynb)

Module index: [02_control_flow/README.md](../README.md)

## 1. Why This Topic Matters

A mission log that silently drops a step, or shifts every timestamp after a dropped reading, is a safety defect rather than a cosmetic one. These built-ins exist because the manual versions of these loops get this wrong.

## 2. Learning Objectives

By the end of this topic you will be able to:

1. Use enumerate instead of a hand-maintained counter and manual indexing.
2. Explain why the stop value of range is exclusive.
3. State why range is lazy and why that matters for large bounds.
4. Predict what zip does when its inputs differ in length.
5. Use strict=True to convert a silent truncation into a loud failure.
6. Derive a time base from the data index so a dropped item leaves a gap.

## 3. Prerequisites

This topic assumes you already have:

- Topic 2.4 Nested Loops and Common Patterns

## 4. Mental Model

**Mental model: three handles onto the same data.** `range` is a counter that can be asked to yield its own index. `enumerate` is the data plus its position, fused into one object. `zip` is several data streams braided into one - but a braid that ends as soon as any strand runs out.

| Built-in | Answers the question | Silent failure |
| --- | --- | --- |
| range | How many times? | Forgetting the stop is exclusive |
| enumerate | Which item is this, and where? | Off-by-one start values |
| zip | What goes with this item? | Stops at the shortest input |

## 5. Terminology

Vocabulary used consistently throughout the course.

| Term | Meaning |
| --- | --- |
| range | A lazy sequence of integers with an exclusive stop. |
| Exclusive stop | range never produces its stop value. |
| enumerate | Iterate an iterable yielding (index, value) pairs. |
| start | The optional first index yielded by enumerate. |
| zip | Iterate several iterables in step, stopping at the shortest. |
| strict | A zip option that raises on a length mismatch. |
| Truncation | Silent loss of the excess items of the longer input. |
| Lazy iterator | An object that produces values on demand. |
| Generator | A function that yields, producing a lazy iterator. |
| Pairing | Attaching a value from one stream to a value from another. |

## 6. Conceptual Explanation

Three built-ins remove almost all the bookkeeping a loop normally needs. `range` generates the index values, so you never maintain a counter by hand. `enumerate` pairs an index with its value, so a loop can use both without indexing. `zip` walks two sequences in step, so related data does not have to be indexed in parallel by hand.

In [ ]:
names = ('ada', 'grace', 'alan')
scores = (0.91, 0.88, 0.95)

print('range   :', list(range(1, 4)))
print('enum    :', list(enumerate(names)))
print('zip     :', list(zip(names, scores)))

The reason to reach for them is not brevity, it is **removing a class of bug**. A hand-maintained counter can drift from the data it is supposed to index. `enumerate` cannot, because the index comes from the same object as the value - which is exactly the bug a rover's logging loop would have when a reading is dropped partway through a batch.

In [ ]:
readings = (0.4, 0.9, 0.7)

for index, reading in enumerate(readings, start=1):
    print(f'sample {index}: {reading}')

> **range is lazy**
>
> range does not build a list. It produces its values on demand, so `range(10**9)` is instant and costs nothing until you iterate it.

## 7. Formal Theory

`range` has three forms. `range(stop)` starts at zero, `range(start, stop)` excludes the stop value, and `range(start, stop, step)` steps by `step`. The stop is always **exclusive**, which is the source of nearly every off-by-one in a range-based loop.

```text
range(stop) == range(0, stop, 1)   and   the stop value is never produced
```

`enumerate` wraps an iterable and yields `(index, value)` pairs, optionally starting from a given number. It is the only way to get an index without counting, and because the index is produced by the same iteration it can never desynchronise from the value.

In [ ]:
print(list(range(0, 10, 3)))
print(list(range(5, 0, -2)))
print(list(range(3)))
print('exclusive stop:', list(range(1, 4)))

`zip` yields tuples drawn from several iterables in step and **stops at the shortest**. That is the safe default - a mismatch truncates rather than crashing - but it is silent, and Python 3.10 added `strict=True` precisely because the silent case causes real defects.

In [ ]:
left = ('a', 'b', 'c')
right = (1, 2)
print('default  :', list(zip(left, right)))
try:
    list(zip(left, right, strict=True))
except ValueError as exc:
    print('strict   : ValueError -', exc)

| Built-in | Yields | Stops when |
| --- | --- | --- |
| range(n) | integers 0 to n-1 | the last value is produced |
| enumerate(s) | (index, value) pairs | the sequence is exhausted |
| zip(a, b) | tuples of one item per input | the shortest input is exhausted |

## 8. Syntax

The canonical forms, each removing a hand-maintained counter.

In [ ]:
readings = (0.4, 0.9, 0.7)

for index in range(len(readings)):
    print(index, readings[index])       # indexing by hand

for index, reading in enumerate(readings):
    print(index, reading)               # the index comes for free

The anti-pattern - zipping against a hand-built index range, which can truncate:

```text
values = (0.4, 0.9, 0.7)
for i, value in zip(range(len(values)), values):
    ...                      # same result, more to get wrong

for i, value in enumerate(values):
    ...                      # the index cannot drift from the value
```

> **zip truncates silently**
>
> zip stops at the shortest input without complaining. If two sequences are meant to correspond, pass strict=True so a mismatch raises instead of quietly discarding data.

## 9. Basic Examples

**Example 1 - indices without a counter.**

In [ ]:
commands = ('stop', 'hold', 'go')
for step, command in enumerate(commands, start=1):
    print(f'step {step}: {command}')

**Example 2 - zipping two logs.**

In [ ]:
timestamps = (0.0, 1.5, 3.0)
battery = (100.0, 92.0, 88.0)
for t, pct in zip(timestamps, battery):
    print(f't={t:>4}s  battery={pct:.1f}%')

## 10. Intermediate Examples

Enumerate is what keeps a log entry attached to the sample it came from, even when the batch is ragged and no single index would work.

In [ ]:
batches = ((0.1, 0.2), (0.3,), (0.4, 0.5, 0.6))

for step, batch in enumerate(batches):
    print(f'batch {step}: {len(batch)} readings')

## 11. Advanced Examples

`zip` is lazy and composable, so it streams rather than materialises. That matters when one side is a generator that should not be fully consumed - for example a live sensor stream paired with a fixed command schedule.

In [ ]:
def readings(count: int):
    for i in range(count):
        yield round(0.5 + i * 0.1, 2)


commands = ('start', 'log', 'stop')
paired = zip(commands, readings(len(commands)))
print('lazy zip:', list(paired))
print('exhausted:', list(paired))

The second call is empty because the generator is now consumed. A lazily zipped stream can only be walked once, which is worth stating in a docstring whenever the result is not materialised immediately.

## 12. Code Walkthrough

Below is a telemetry log builder. It pairs a timestamp sequence with a reading sequence, attaches 1-based step numbers, and refuses to build a log from mismatched data rather than silently truncating it.

In [ ]:
"""telemetry.py - build a timestamped, step-numbered log."""
from __future__ import annotations

SAMPLE_PERIOD_S = 0.5


def build_log(readings, period_s: float = SAMPLE_PERIOD_S, strict: bool = True) -> list[dict]:
    """Return one dict per reading: step, t_s and value.

    strict=True raises when the reading sequence contains a
    non-numeric entry rather than quietly dropping it, because a
    log that loses a sample is worse than one that fails loudly.
    """
    log: list[dict] = []
    for step, (t_s, value) in enumerate(
        zip((index * period_s for index in range(len(readings))), readings, strict=strict)
    ):
        if not isinstance(value, (int, float)) or isinstance(value, bool):
            if strict:
                raise TypeError(f'step {step}: non-numeric reading {value!r}')
            continue
        log.append({'step': step, 't_s': round(t_s, 3), 'value': value})
    return log

Two choices matter here. The timestamps come from a generator sized to the readings, so they can never be longer or shorter, and the numeric check is policy rather than accident: a malformed sample either raises or is skipped, and which one happens is stated in the docstring rather than left to chance. Together these remove the two ways a log of this kind usually goes quietly wrong, which is a truncated tail and a time base that drifts once a sample is dropped.

## 13. Line-by-Line Explanation

1. SAMPLE_PERIOD_S is a constant so the logging cadence is auditable without reading the loop.
2. The timestamp generator is sized from len(readings), so it can never desynchronise from the values being logged.
3. strict on zip is the default here because a truncated log is a silent defect, and a loud one is preferable.
4. enumerate starts at zero, so the step number and the list index agree - adding 1 later would be a separate, explicit decision.
5. The type check is placed before the append so a bad sample never reaches the log.
6. strict decides whether a bad sample raises or is skipped, and the docstring states which, so the caller is never guessing.

## 14. Common Mistakes

**Mistake 1 - a range that drops the last element.**

```text
values = (1, 2, 3)
for i in range(len(values)):
    ...            # correct, but awkward next to enumerate

for i, value in enumerate(values):
    ...            # the same three items, no counting
```

**Mistake 2 - assuming range includes its stop value.**

```text
list(range(1, 4))    # [1, 2, 3] - 4 is NOT included
len(range(1, 4))     # 3, not 4
range(0, 10, 3)      # [0, 3, 6, 9] - stops before exceeding 10
```

**Mistake 3 - zip truncating without complaint.**

```text
left = ('a', 'b', 'c')
right = (1, 2)
list(zip(left, right))                  # [(a,1), (b,2)] - c is lost
list(zip(left, right, strict=True))     # raises ValueError
```

**Mistake 4 - a lazy zip consumed twice.**

```text
paired = zip(a, b)
list(paired)     # the first walk consumes the inputs
list(paired)     # empty the second time
```

## 15. Debugging Techniques

When a range-based loop visits the wrong elements, print the range itself. Seeing the exact sequence removes any guesswork about the exclusive stop.

In [ ]:
print('range(1, 4) ->', list(range(1, 4)))
print('range(0, 10, 3) ->', list(range(0, 10, 3)))
print('len', len(range(1, 4)))

When a zip loses data, compare the lengths before trusting the pairing. A mismatch is the whole explanation in one line.

In [ ]:
left = ('a', 'b', 'c')
right = (1, 2)
print('lengths:', len(left), len(right))
print('paired :', list(zip(left, right)))
try:
    list(zip(left, right, strict=True))
except ValueError as exc:
    print('strict  :', exc)

> **len() on a range is O(1)**
>
> range knows its own length without being materialised, so comparing len(range(n)) with len(sequence) is free even for very large n.

## 16. Best Practices

- Prefer enumerate over range(len(seq)) and indexing.
- Use range for counting, not for indexing a sequence.
- Remember the stop value of range is exclusive.
- Pass strict=True to zip when lengths are meant to correspond.
- Materialise a lazy zip once, and do not walk it twice.
- Generate the second side of a zip rather than slicing it.
- Keep enumerate and zip together readable; two levels of nesting is the limit.
- Name the pairing in a variable rather than repeating the zip call.

## 17. Performance Considerations

All three built-ins are lazy, which is the performance point that matters. `range(10**9)` allocates nothing; it is a small object that computes each value when asked. `zip` likewise holds no intermediate list, so pairing two ten-million-element sequences costs one tuple per step rather than a materialised result.

In [ ]:
big = range(10**9)
print('len is O(1):', len(big))
print('first three :', [next(iter(big))] and list(big)[:3])

a = range(1_000_000)
b = range(1_000_000)
total = sum(1 for pair in zip(a, b) if pair[0] == pair[1])
print('matching pairs:', total)

The practical consequence is that you should not materialise these results unless you need to. Wrapping a lazy generator in list() to "be safe" both costs memory and forecloses a second traversal, which is almost never what was wanted.

> **islice for early termination**
>
> itertools.islice walks only as much of a lazy iterator as you ask for, so taking the first n zip pairs costs n steps rather than the whole sequence.

## 18. Pythonic Approaches

Three hand-rolled loops against their built-in equivalents.

In [ ]:
values = (0.4, 0.9, 0.7)

# before: a counter maintained alongside the data
index = 0
for value in values:
    print(index, value)
    index += 1

# after: the index comes from the iteration
for index, value in enumerate(values):
    print(index, value)

## 19. Robotics Connection

Pairing a rover's live battery trace with a fixed command schedule is exactly what zip is for - and exactly where silent truncation would be a safety problem rather than a curiosity.

In [ ]:
import sys
from pathlib import Path

here = Path.cwd()
root = next((p for p in [here, *here.parents] if (p / 'shared').is_dir()), None)
if root is not None:
    sys.path.insert(0, str(root))

from shared.robo_x_sim import SimulatedRobot  # noqa: E402

robot = SimulatedRobot(name='rover-01', battery_wh=48.0)
schedule = ('start', 'log', 'hold', 'stop')
battery = (robot.status()['battery_pct'] for _ in schedule)

for step, (command, pct) in enumerate(zip(schedule, battery, strict=True), 1):
    print(f'step {step}: {command:<6} battery {pct:.1f}%')

## 20. Engineering Example

A range is the right tool when you genuinely want indices - for example to produce move commands of a known length, where no sequence is being indexed.

In [ ]:
def retry_delays(attempts: int, base_s: float = 0.5):
    """Exponential backoff as a range-driven generator."""
    for attempt in range(1, attempts + 1):
        yield round(base_s * (2 ** (attempt - 1)), 3)

## 21. Guided Practice

1. Print range(1, 4) and confirm the stop is excluded.
2. Loop over a tuple with enumerate and print 1-based step numbers.
3. Zip a three-item sequence with a two-item one and observe the truncation, then repeat with strict=True.
4. Build a timestamp sequence with a generator and pair it with a reading list.

In [ ]:
print('range(1, 4):', list(range(1, 4)))

for step, name in enumerate(('ada', 'grace'), start=1):
    print('step', step, name)

print('zip default:', list(zip((1, 2, 3), ('a', 'b'))))
try:
    list(zip((1, 2, 3), ('a', 'b'), strict=True))
except ValueError as exc:
    print('zip strict :', exc)

times = (i * 0.5 for i in range(3))
for t, value in zip(times, (0.1, 0.2, 0.3), strict=True):
    print('log', t, value)

## 22. Knowledge Checks

Answer these before moving on. Full answers are in [solution.ipynb](solution.ipynb).

- **Q1.** Which statement about range is true?
- **Q1.** Why prefer `enumerate(seq)` over `range(len(seq))`?
- **Q1.** What does `zip(a, b, strict=True)` do on a length mismatch?
- **Q1.** Why must the timestamp come from the index rather than from a counter of kept readings?

## 23. Summary

`range`, `enumerate` and `zip` are not shorthand for the bookkeeping a loop normally needs - they remove the bookkeeping itself. A hand-maintained counter can drift from the data it indexes, whereas the index `enumerate` yields comes from the same iteration as the value, so the two cannot disagree. That is the argument for the built-in over the shorter-looking manual loop.

Each carries one silent failure, and knowing it is the practical skill. `range` has an **exclusive stop**, so `range(1, 4)` yields three values rather than four, and an off-by-one there quietly drops the last element. `zip` stops at the **shortest** input, which is the safe default but discards the excess without complaint; `strict=True` turns that silent truncation into a `ValueError` that says exactly what was lost. And because all three are lazy, a result built with `zip` can be walked only once unless it is materialised first.

For robotics work the difference is practical rather than stylistic. Pairing a rover's live battery trace with a fixed command schedule is what `zip` is for, and a truncated pairing in a mission log is a safety defect rather than an inconvenience. Generating the second stream - so its length is derived from the first - removes the mismatch instead of merely detecting it.

## 24. Key Takeaways

- Prefer enumerate over range(len(seq)) and indexing.
- Use range for counting, never for indexing a sequence.
- range stops exclusively; range(1, 4) has three values.
- zip stops at the shortest input, silently.
- Pass strict=True to zip when lengths must correspond.
- A lazy zip can be walked once; materialise it deliberately.
- Generate the second stream so lengths cannot disagree.
- Use zip to attach a value, not to build a list by hand.

## 25. Further Exploration

Read further: the official documentation for [`range`](https://docs.python.org/3/library/stdtypes.html#range), [`enumerate`](https://docs.python.org/3/library/functions.html#enumerate) and [`zip`](https://docs.python.org/3/library/functions.html#zip), plus [`itertools.islice`](https://docs.python.org/3/library/itertools.html#itertools.islice) for taking a prefix of a lazy iterator.